In [1]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 29 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (579 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
!ollama --version

In [4]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("Ollama server started successfully")

Ollama server started successfully


In [5]:
!ollama pull llama3.2:3b

In [6]:
!ollama list

NAME           ID              SIZE      MODIFIED       
llama3.2:3b    a80c4f17acd5    2.0 GB    20 seconds ago    


In [7]:
!pip install -q langchain langchain-core langchain-ollama

In [8]:
from langchain_core.prompts import PromptTemplate
from langchain_ollama import OllamaLLM
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

print("LangChain imported successfully")

LangChain imported successfully


In [9]:
llm = OllamaLLM(
    model="llama3.2:3b",
    base_url="http://localhost:11434"
)

print("Ollama LLM ready")

Ollama LLM ready


In [10]:
prompt = PromptTemplate(
    input_variables=["topic"],
    template="""
Explain {topic} in simple terms.
Give only 2-3 short sentences.
"""

)

parser = StrOutputParser()

chain = prompt | llm | parser

print("Prompt → Ollama → OutputParser chain created")

Prompt → Ollama → OutputParser chain created


In [11]:
topics = [
    "Artificial Intelligence",
    "Machine Learning",
    "Deep Learning",
    "Generative AI",
    "Large Language Models"
]

for i, topic in enumerate(topics, 1):
    result = chain.invoke({"topic": topic})

    print(f"\nTest {i}: {topic}")
    print(result)
    print("-" * 40)


Test 1: Artificial Intelligence
Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making. These systems use complex algorithms and data to mimic human behavior, allowing them to improve and adapt over time. AI can be used in various applications, such as virtual assistants, image recognition, and autonomous vehicles.
----------------------------------------

Test 2: Machine Learning
Machine Learning is a type of artificial intelligence that allows computers to learn from data without being explicitly programmed. It enables computers to make predictions, classify objects, and make decisions based on patterns in the data they've been trained on. Think of it like teaching a computer to recognize a cat by showing it many pictures of cats, so it can learn to identify them on its own.
----------------------------------------

Test 3: Deep Learning
Deep L

In [12]:
conversation_history = []

print("Conversation memory initialized")

Conversation memory initialized


In [13]:
memory_prompt = PromptTemplate(
    input_variables=["history", "input"],
    template="""
You are a helpful assistant.

Conversation history:
{history}

Current user message:
{input}

Answer using the conversation history when useful.
Keep the answer to 2-3 short sentences.
"""
)

memory_chain = memory_prompt | llm | parser

print("Memory chain created")

Memory chain created


In [14]:
def chat_with_memory(user_input):

    history_text = "\n".join(
        [
            f"User: {msg.content}" if isinstance(msg, HumanMessage)
            else f"AI: {msg.content}"
            for msg in conversation_history
        ]
    )

    response = memory_chain.invoke({
        "history": history_text,
        "input": user_input
    })

    conversation_history.append(
        HumanMessage(content=user_input)
    )

    conversation_history.append(
        AIMessage(content=response)
    )

    return response

In [15]:
turns = [
    "My name is Bhavana.",
    "I am an engineering student.",
    "What am I studying?",
    "What is my name?",
    "Suggest a simple AI project for me."
]

for i, message in enumerate(turns, 1):

    response = chat_with_memory(message)

    print(f"\nTurn {i}")
    print("User:", message)
    print("AI:", response)
    print("-" * 40)


Turn 1
User: My name is Bhavana.
AI: Welcome Bhavana! It's nice to meet you. What brings you here today?
----------------------------------------

Turn 2
User: I am an engineering student.
AI: Nice to meet you, Bhavana! As an engineering student, I'm sure you're juggling a lot of responsibilities and information. How can I assist you today?
----------------------------------------

Turn 3
User: What am I studying?
AI: Bhavana, it seems you're an engineering student looking for a refresher. You're currently studying Computer Science and Engineering, with a strong focus on algorithms and data structures. Is there something specific you'd like to know or discuss about this subject?
----------------------------------------

Turn 4
User: What is my name?
AI: I recall that your name is Bhavana, and you're an engineering student studying Computer Science and Engineering. Is there something specific you'd like to know or discuss about algorithms and data structures?
--------------------------

In [16]:
print("===== Conversation History =====")

for message in conversation_history:

    if isinstance(message, HumanMessage):
        print("User:", message.content)
    else:
        print("AI:", message.content)

    print()

===== Conversation History =====
User: My name is Bhavana.

AI: Welcome Bhavana! It's nice to meet you. What brings you here today?

User: I am an engineering student.

AI: Nice to meet you, Bhavana! As an engineering student, I'm sure you're juggling a lot of responsibilities and information. How can I assist you today?

User: What am I studying?

AI: Bhavana, it seems you're an engineering student looking for a refresher. You're currently studying Computer Science and Engineering, with a strong focus on algorithms and data structures. Is there something specific you'd like to know or discuss about this subject?

User: What is my name?

AI: I recall that your name is Bhavana, and you're an engineering student studying Computer Science and Engineering. Is there something specific you'd like to know or discuss about algorithms and data structures?

User: Suggest a simple AI project for me.

AI: Considering your background in Computer Science and Engineering, I think a simple project lik

In [17]:
from langchain_core.tools import tool

@tool
def web_search(query: str) -> str:
    """Simulated web search tool."""
    return f"Search result for '{query}': This is a simulated web search result."


@tool
def calculator(expression: str) -> str:
    """Calculator tool."""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"


print("Two tools created:")
print("- Web Search")
print("- Calculator")

Two tools created:
- Web Search
- Calculator


In [18]:
print("Web Search:")
print(web_search.invoke("Generative AI"))

print("\nCalculator:")
print(calculator.invoke("25 * 4"))

Web Search:
Search result for 'Generative AI': This is a simulated web search result.

Calculator:
100


In [19]:
def simple_agent(task):

    task_lower = task.lower()

    if any(symbol in task_lower for symbol in ["calculate", "*", "/", "+", "-"]):

        expression = task_lower.replace("calculate", "").strip()

        result = calculator.invoke(expression)

        return f"Calculator result: {result}"

    else:

        result = web_search.invoke(task)

        return f"Search result: {result}"

In [20]:
tasks = [
    "Calculate 25 * 8",
    "Calculate 500 / 20",
    "Search for Generative AI applications"
]

for i, task in enumerate(tasks, 1):

    print(f"\nTask {i}")
    print("Input:", task)
    print("Output:", simple_agent(task))
    print("-" * 40)


Task 1
Input: Calculate 25 * 8
Output: Calculator result: 200
----------------------------------------

Task 2
Input: Calculate 500 / 20
Output: Calculator result: 25.0
----------------------------------------

Task 3
Input: Search for Generative AI applications
Output: Search result: Search result for 'Search for Generative AI applications': This is a simulated web search result.
----------------------------------------
